# Afrikaans MT Benchmark — mbart-large-50

Model: `facebook/mbart-large-50-many-to-many-mmt`  
Dataset: Tatoeba Afrikaans–English (public, Helsinki-NLP)  
Direction: `af_ZA` → `en_XX`

See [`benchmarks/af/README.md`](README.md) for evaluation notes.

## Setup

**Google Colab:** run the install cell.  
**Local:** `pip install transformers sentencepiece bert_score nltk`

In [ ]:
%%capture
!pip install transformers sentencepiece bert_score sacrebleu nltk accelerate

import nltk
nltk.download('wordnet')
nltk.download('punkt_tab')

In [ ]:
import torch
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader, Dataset
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
from bert_score import score as bert_score
import nltk

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.data[idx], return_tensors="pt",
            padding="max_length", truncation=True, max_length=self.max_length
        )
        return {k: v.squeeze() for k, v in encoded.items()}

## Load Afrikaans–English benchmark data

In [ ]:
# Tatoeba Afrikaans–English test data (public, Helsinki-NLP)
# https://github.com/Helsinki-NLP/Tatoeba-Challenge
# Not used in mbart-large-50 training split

df = pd.DataFrame({
    'source': [
        'Ek haat om te wag.',
        'Ek wil weet wie het hierdie venster gebreek.',
        'Jy is baie laat.',
        'Ek sal terugkom na jou oor Tom.',
        "Sy gaan 'n bietjie koffie maak.",
    ],
    'target': [
        'I hate waiting.',
        'I want to know who broke this window.',
        "You're very late.",
        "I'll get back to you about Tom.",
        "She's going to make some coffee.",
    ]
})
print(f'Dataset: {len(df)} rows')
df.head()

## Translate — mbart-large-50

In [ ]:
MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt"
SRC_LANG   = "af_ZA"
TGT_LANG   = "en_XX"
BATCH_SIZE = 4

model     = MBartForConditionalGeneration.from_pretrained(MODEL_NAME)
tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)
tokenizer.src_lang = SRC_LANG

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print(f'Running on: {device}')

dataset    = TranslationDataset(df['source'].tolist(), tokenizer)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)
translations = []

for batch in tqdm(dataloader):
    data = {k: v.to(device) for k, v in batch.items()}
    tokens = model.generate(**data, forced_bos_token_id=tokenizer.lang_code_to_id[TGT_LANG])
    translations.extend(tokenizer.batch_decode(tokens, skip_special_tokens=True))

df['mbart_translation'] = translations
df[['source', 'target', 'mbart_translation']]

## Evaluate — METEOR

In [ ]:
meteor = nltk.translate.meteor_score.meteor_score
scores = []
for ref, hyp in tqdm(zip(df['target'], df['mbart_translation'])):
    score = meteor([ref.split()], hyp.split())
    scores.append(score)

df['meteor'] = scores
print(f"Average METEOR: {np.mean(scores):.4f}")
df[['source','target','translation','meteor']].head(10)

## Evaluate — BERTScore

In [ ]:
P, R, F1 = bert_score(df['mbart_translation'].tolist(), df['target'].tolist(), lang='en', verbose=True)
df['bertscore'] = F1.numpy()
print(f"Average BERTScore F1: {F1.mean():.4f}")

## Error analysis

In [ ]:
# Summary statistics
summary = pd.DataFrame({
    'bertscore': df['bertscore'].describe(),
    'meteor':    df['meteor'].describe()
})
print(summary)

# Sample low-scoring translations for manual review
low = df[df['bertscore'] < df['bertscore'].quantile(0.25)].sample(min(10, len(df)))
for _, row in low.iterrows():
    print(f"\nSOURCE:      {row['source']}")
    print(f"GOLD:        {row['target']}")
    print(f"TRANSLATION: {row['translation']}")
    print(f"METEOR: {row['meteor']:.3f}  BERTScore: {row['bertscore']:.3f}")